# Download and prepare the three Fed-MOPSD data domains

This CPU-only notebook rebuilds the same data sources used by `FedAvg_mopsd.ipynb` under `/code/data`. It creates deterministic 3,000-row normalized training files (the old experiment consumed their first 2,000 rows) and downloads the complete evaluation sets. Completed outputs and interrupted `.part` downloads are reused.


In [ ]:
# Papermill parameters
DATA_ROOT = "data"
TRAIN_ROWS = 3000
HF_ENDPOINT = "https://huggingface.co"
DOWNLOAD_ENDPOINTS = ["https://hf-mirror.com", "https://huggingface.co"]
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 120
MAX_ATTEMPTS = 100
PROGRESS_INTERVAL_SECONDS = 5
CHUNK_SIZE_MIB = 1


## Runtime and output directories


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT.rstrip("/")

import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from datasets import DownloadConfig, load_dataset
from huggingface_hub import HfApi

data_root = Path(DATA_ROOT).expanduser().resolve()
cache_root = data_root / ".hf_cache"
for domain in ("math", "if", "code"):
    (data_root / domain / "train").mkdir(parents=True, exist_ok=True)
    (data_root / domain / "eval").mkdir(parents=True, exist_ok=True)
cache_root.mkdir(parents=True, exist_ok=True)

TRAIN_SOURCES = {
    "math": {
        "dataset": "siyanzhao/Openthoughts_math_30k_opsd",
        "config": None,
        "split": "train",
        "revision": "1f33e9dc2e8a1c639ca74f8024ad4a9f1f5eae62",
    },
    "if": {
        "dataset": "allenai/tulu-3-sft-personas-instruction-following",
        "config": None,
        "split": "train",
        "revision": "fe0c7d350c9b4542b8d829a6f1daa1c259f0ba0e",
    },
    "code": {
        "dataset": "microsoft/rStar-Coder",
        "config": "seed_sft",
        "split": "train",
        "revision": "3a7a0a0636ec96e3c1ec42ebe79ade467caa040d",
    },
}

print(f"Data root: {data_root}", flush=True)
print(f"CPU-only: CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']!r}", flush=True)


## Reusable download and serialization helpers


In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def format_duration(seconds):
    if seconds is None or seconds == float("inf"):
        return "--:--:--"
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def parquet_complete(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size < 8:
        return False
    with path.open("rb") as handle:
        if handle.read(4) != b"PAR1":
            return False
        handle.seek(-4, os.SEEK_END)
        return handle.read(4) == b"PAR1"


def response_total_size(response, resume_from):
    match = re.search(r"/(\d+)$", response.headers.get("Content-Range", ""))
    if match:
        return int(match.group(1))
    length = response.headers.get("Content-Length")
    if length and length.isdigit():
        return int(length) + (resume_from if response.status_code == 206 else 0)
    return None


def download_repo_parquet(repo_id, revision, remote_path, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    part_path = destination.with_name(destination.name + ".part")
    if parquet_complete(destination):
        print(f"[REUSE] {destination}: {destination.stat().st_size / 2**20:.2f} MiB", flush=True)
        return destination
    if destination.exists():
        destination.unlink()

    session = requests.Session()
    session.headers.update({
        "User-Agent": "fed-mopsd-data-downloader/1.0",
        "Accept-Encoding": "identity",
    })
    started = time.time()
    missing_endpoints = set()
    for attempt in range(1, int(MAX_ATTEMPTS) + 1):
        resume_from = part_path.stat().st_size if part_path.exists() else 0
        endpoint = DOWNLOAD_ENDPOINTS[(attempt - 1) % len(DOWNLOAD_ENDPOINTS)].rstrip("/")
        url = f"{endpoint}/datasets/{repo_id}/resolve/{revision}/{remote_path}?download=true"
        headers = {"Range": f"bytes={resume_from}-"} if resume_from else {}
        print(
            f"[CONNECT] {destination.name}: attempt={attempt}/{MAX_ATTEMPTS} "
            f"resume_from={resume_from / 2**20:.2f} MiB endpoint={endpoint}",
            flush=True,
        )
        try:
            with session.get(
                url, headers=headers, stream=True, allow_redirects=True,
                timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
            ) as response:
                if response.status_code == 416 and parquet_complete(part_path):
                    os.replace(part_path, destination)
                    return destination
                response.raise_for_status()
                if resume_from and response.status_code == 200:
                    print("[RESTART] Server ignored Range; restarting this file.", flush=True)
                    part_path.unlink(missing_ok=True)
                    resume_from = 0
                expected_total = response_total_size(response, resume_from)
                last_report = time.time()
                with part_path.open("ab" if resume_from else "wb") as handle:
                    for chunk in response.iter_content(chunk_size=int(CHUNK_SIZE_MIB * 2**20)):
                        if not chunk:
                            continue
                        handle.write(chunk)
                        now = time.time()
                        if now - last_report >= PROGRESS_INTERVAL_SECONDS:
                            downloaded = part_path.stat().st_size
                            elapsed = max(now - started, 1e-6)
                            speed = downloaded / elapsed
                            if expected_total:
                                percent = 100.0 * downloaded / expected_total
                                total_text = f"/{expected_total / 2**20:.2f} MiB ({percent:.2f}%)"
                                eta = (expected_total - downloaded) / speed if speed else None
                            else:
                                total_text, eta = " MiB", None
                            print(
                                f"[PROGRESS] {destination.name}: {downloaded / 2**20:.2f}{total_text} "
                                f"speed={speed / 2**20:.2f} MiB/s elapsed={format_duration(elapsed)} "
                                f"eta={format_duration(eta)}",
                                flush=True,
                            )
                            last_report = now
                    handle.flush()
                    os.fsync(handle.fileno())
                downloaded = part_path.stat().st_size
                if expected_total is not None and downloaded < expected_total:
                    raise IOError(f"incomplete response: {downloaded}/{expected_total} bytes")
                if not parquet_complete(part_path):
                    raise IOError("downloaded file is not a complete Parquet file")
                os.replace(part_path, destination)
                print(f"[DONE] {destination}: {destination.stat().st_size / 2**20:.2f} MiB", flush=True)
                return destination
        except (requests.RequestException, OSError) as exc:
            status = exc.response.status_code if isinstance(exc, requests.HTTPError) and exc.response is not None else None
            if status == 404:
                missing_endpoints.add(endpoint)
                if len(missing_endpoints) >= len(DOWNLOAD_ENDPOINTS):
                    raise FileNotFoundError(f"Missing pinned file: {repo_id}@{revision}/{remote_path}") from exc
                continue
            if status is not None and 400 <= status < 500 and status not in (408, 429):
                raise RuntimeError(f"Non-retryable HTTP {status}: {remote_path}") from exc
            if attempt >= int(MAX_ATTEMPTS):
                raise RuntimeError(f"Failed after {attempt} attempts: {remote_path}") from exc
            wait_seconds = min(60, max(2, attempt * 2))
            saved = part_path.stat().st_size / 2**20 if part_path.exists() else 0.0
            print(f"[RETRY] {type(exc).__name__}: {exc}; saved={saved:.2f} MiB; wait={wait_seconds}s", flush=True)
            time.sleep(wait_seconds)
    raise RuntimeError(f"Unreachable retry state: {remote_path}")


def jsonl_is_complete(path, expected_rows, domain):
    path = Path(path)
    if not path.is_file():
        return False
    count = 0
    try:
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                row = json.loads(line)
                if row.get("domain") != domain or not row.get("question") or not row.get("reference_answer"):
                    return False
                count += 1
    except (OSError, json.JSONDecodeError):
        return False
    return count == expected_rows


def write_jsonl_atomic(path, rows):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_manifest(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temporary, path)


## Math training set — pinned OPSD Math 30K


In [ ]:
math_train_dir = data_root / "math" / "train"
math_raw_dir = math_train_dir / "raw"
math_raw_dir.mkdir(parents=True, exist_ok=True)
math_remote_files = [
    "data/train-00000-of-00002.parquet",
    "data/train-00001-of-00002.parquet",
]
math_source = TRAIN_SOURCES["math"]
math_raw_paths = [
    download_repo_parquet(
        math_source["dataset"], math_source["revision"], remote_path,
        math_raw_dir / Path(remote_path).name,
    )
    for remote_path in math_remote_files
]

math_train_path = math_train_dir / "train.jsonl"
math_manifest_path = math_train_dir / "manifest.json"
if jsonl_is_complete(math_train_path, int(TRAIN_ROWS), "math"):
    print(f"[REUSE] Math normalized training set: {math_train_path}", flush=True)
else:
    dataset = load_dataset(
        "parquet", data_files=[str(path) for path in math_raw_paths], split="train",
        cache_dir=str(cache_root / "datasets"),
    )
    selected, scanned = [], 0
    for row in dataset:
        scanned += 1
        if row.get("correct") is False:
            continue
        question = str(row.get("problem") or "").strip()
        answer = str(row.get("solution") or "").strip()
        if not question or not answer:
            continue
        selected.append({
            "sample_id": hashlib.sha256(question.encode("utf-8")).hexdigest()[:20],
            "domain": "math",
            "question": question,
            "reference_answer": answer,
            "selection_index": len(selected),
        })
        if len(selected) >= int(TRAIN_ROWS):
            break
    if len(selected) != int(TRAIN_ROWS):
        raise RuntimeError(f"Math produced only {len(selected)}/{TRAIN_ROWS} usable rows")
    write_jsonl_atomic(math_train_path, selected)
    write_manifest(math_manifest_path, {
        "domain": "math", "semantic_role": "train", "rows": len(selected),
        "scanned_rows": scanned, "source": math_source,
        "filter": "correct is not False", "selection": "first-valid-records",
        "output": str(math_train_path), "created_at_utc": utc_now(),
    })
    print(f"[DONE] Math training: {len(selected)} rows from {scanned} scanned rows", flush=True)
if "dataset" in locals():
    del dataset


## IF training set — pinned Tulu personas instruction-following


In [ ]:
def assistant_message_text(messages):
    if not isinstance(messages, list):
        return ""
    for message in reversed(messages):
        if isinstance(message, dict) and message.get("role") == "assistant":
            return str(message.get("content") or "").strip()
    return ""


if_train_dir = data_root / "if" / "train"
if_train_path = if_train_dir / "train.jsonl"
if_manifest_path = if_train_dir / "manifest.json"
if_source = TRAIN_SOURCES["if"]
if jsonl_is_complete(if_train_path, int(TRAIN_ROWS), "if"):
    print(f"[REUSE] IF normalized training set: {if_train_path}", flush=True)
else:
    stream = load_dataset(
        if_source["dataset"], split=if_source["split"], revision=if_source["revision"],
        streaming=True, cache_dir=str(cache_root / "datasets"),
    )
    selected, scanned = [], 0
    for row in stream:
        scanned += 1
        question = str(row.get("prompt") or "").strip()
        answer = assistant_message_text(row.get("messages"))
        if not question or not answer:
            continue
        selected.append({
            "sample_id": str(row.get("id") or hashlib.sha256(question.encode("utf-8")).hexdigest()[:20]),
            "domain": "if",
            "question": question,
            "reference_answer": answer,
            "selection_index": len(selected),
        })
        if len(selected) % 500 == 0:
            print(f"[PROGRESS] IF train: selected={len(selected)}/{TRAIN_ROWS} scanned={scanned}", flush=True)
        if len(selected) >= int(TRAIN_ROWS):
            break
    if len(selected) != int(TRAIN_ROWS):
        raise RuntimeError(f"IF produced only {len(selected)}/{TRAIN_ROWS} usable rows")
    write_jsonl_atomic(if_train_path, selected)
    write_manifest(if_manifest_path, {
        "domain": "if", "semantic_role": "train", "rows": len(selected),
        "scanned_rows": scanned, "source": if_source,
        "filter": "non-empty prompt and assistant response",
        "selection": "first-valid-records", "output": str(if_train_path),
        "created_at_utc": utc_now(),
    })
    print(f"[DONE] IF training: {len(selected)} rows from {scanned} scanned rows", flush=True)
if "stream" in locals():
    del stream


## Code training set — pinned rStar-Coder `seed_sft`


In [ ]:
code_train_dir = data_root / "code" / "train"
code_train_path = code_train_dir / "train.jsonl"
code_manifest_path = code_train_dir / "manifest.json"
code_source = TRAIN_SOURCES["code"]
if jsonl_is_complete(code_train_path, int(TRAIN_ROWS), "code"):
    print(f"[REUSE] Code normalized training set: {code_train_path}", flush=True)
else:
    stream = load_dataset(
        code_source["dataset"], code_source["config"], split=code_source["split"],
        revision=code_source["revision"], streaming=True,
        cache_dir=str(cache_root / "datasets"),
    )
    selected, scanned = [], 0
    for row in stream:
        scanned += 1
        if not (row.get("verified") is True and row.get("is_passed") is True):
            if scanned % 5000 == 0:
                print(f"[PROGRESS] Code train: selected={len(selected)}/{TRAIN_ROWS} scanned={scanned}", flush=True)
            continue
        question = str(row.get("question") or "").strip()
        answer = str(row.get("response") or "").strip()
        if not question or not answer:
            continue
        selected.append({
            "sample_id": str(row.get("question_id") or hashlib.sha256(question.encode("utf-8")).hexdigest()[:20]),
            "domain": "code",
            "question": question,
            "reference_answer": answer,
            "selection_index": len(selected),
        })
        if len(selected) % 500 == 0:
            print(f"[PROGRESS] Code train: selected={len(selected)}/{TRAIN_ROWS} scanned={scanned}", flush=True)
        if len(selected) >= int(TRAIN_ROWS):
            break
    if len(selected) != int(TRAIN_ROWS):
        raise RuntimeError(f"Code produced only {len(selected)}/{TRAIN_ROWS} usable rows")
    write_jsonl_atomic(code_train_path, selected)
    write_manifest(code_manifest_path, {
        "domain": "code", "semantic_role": "train", "rows": len(selected),
        "scanned_rows": scanned, "source": code_source,
        "filter": "verified == True and is_passed == True",
        "selection": "first-valid-records", "output": str(code_train_path),
        "created_at_utc": utc_now(),
    })
    print(f"[DONE] Code training: {len(selected)} rows from {scanned} scanned rows", flush=True)
if "stream" in locals():
    del stream


## Math and IF evaluation sets

AIME25/AIME26 expose `test`; IFBench/IFEval expose their evaluation data under a split named `train`. Each repository's resolved commit is stored in its manifest.


In [ ]:
api = HfApi(endpoint=HF_ENDPOINT.rstrip("/"))
download_config = DownloadConfig(cache_dir=str(cache_root / "downloads"), resume_download=True)

EVAL_SOURCES = [
    {"domain": "math", "name": "aime25", "dataset": "math-ai/aime25", "split": "test",
     "required": {"problem", "answer"}},
    {"domain": "math", "name": "aime26", "dataset": "math-ai/aime26", "split": "test",
     "required": {"problem", "answer"}},
    {"domain": "if", "name": "ifbench", "dataset": "allenai/IFBench_test", "split": "train",
     "required": {"key", "prompt", "instruction_id_list", "kwargs"}},
    {"domain": "if", "name": "ifeval", "dataset": "google/IFEval", "split": "train",
     "required": {"key", "prompt", "instruction_id_list", "kwargs"}},
]

eval_manifests = {"math": [], "if": []}
for source in EVAL_SOURCES:
    output_path = data_root / source["domain"] / "eval" / f"{source['name']}.parquet"
    manifest_path = output_path.with_suffix(".manifest.json")
    if parquet_complete(output_path) and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"[REUSE] {source['name']}: {manifest.get('rows')} rows at {output_path}", flush=True)
        eval_manifests[source["domain"]].append(manifest)
        continue
    resolved_revision = api.dataset_info(source["dataset"]).sha
    print(f"[LOAD] {source['name']}: {source['dataset']}@{resolved_revision}", flush=True)
    dataset = load_dataset(
        source["dataset"], split=source["split"], revision=resolved_revision,
        cache_dir=str(cache_root / "datasets"), download_config=download_config,
    )
    missing = source["required"] - set(dataset.column_names)
    if missing:
        raise RuntimeError(f"{source['name']} is missing columns: {sorted(missing)}")
    temporary = output_path.with_name(output_path.name + ".tmp.parquet")
    dataset.to_parquet(str(temporary))
    if not parquet_complete(temporary):
        raise RuntimeError(f"Failed to create a valid local Parquet file: {temporary}")
    os.replace(temporary, output_path)
    manifest = {
        "domain": source["domain"], "semantic_role": "eval",
        "name": source["name"], "dataset": source["dataset"],
        "split": source["split"], "revision": resolved_revision,
        "rows": len(dataset), "columns": dataset.column_names,
        "output": str(output_path), "created_at_utc": utc_now(),
    }
    write_manifest(manifest_path, manifest)
    eval_manifests[source["domain"]].append(manifest)
    print(f"[DONE] {source['name']}: {len(dataset)} rows -> {output_path}", flush=True)

for domain in ("math", "if"):
    write_manifest(data_root / domain / "eval" / "manifest.json", {
        "domain": domain, "semantic_role": "eval",
        "datasets": eval_manifests[domain], "updated_at_utc": utc_now(),
    })


## Code evaluation set — pinned LiveCodeBench release_v6


In [ ]:
LCB_REPO = "livecodebench/code_generation_lite"
LCB_REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"
LCB_RELEASE = "release_v6"
code_eval_dir = data_root / "code" / "eval"
lcb_paths = []
for shard_index in range(9):
    filename = f"test-{shard_index:05d}-of-00009.parquet"
    lcb_paths.append(download_repo_parquet(
        LCB_REPO, LCB_REVISION, f"{LCB_RELEASE}/{filename}", code_eval_dir / filename,
    ))

write_manifest(code_eval_dir / "manifest.json", {
    "domain": "code", "semantic_role": "eval",
    "dataset": LCB_REPO, "release": LCB_RELEASE, "revision": LCB_REVISION,
    "files": [{"name": path.name, "bytes": path.stat().st_size} for path in lcb_paths],
    "total_bytes": sum(path.stat().st_size for path in lcb_paths),
    "updated_at_utc": utc_now(),
})
print(f"[DONE] LiveCodeBench: 9/9 shards, {sum(p.stat().st_size for p in lcb_paths) / 2**30:.3f} GiB", flush=True)


## Final verification


In [ ]:
expected_training = {
    domain: data_root / domain / "train" / "train.jsonl"
    for domain in ("math", "if", "code")
}
for domain, path in expected_training.items():
    if not jsonl_is_complete(path, int(TRAIN_ROWS), domain):
        raise RuntimeError(f"Incomplete {domain} training output: {path}")

expected_eval = [
    data_root / "math" / "eval" / "aime25.parquet",
    data_root / "math" / "eval" / "aime26.parquet",
    data_root / "if" / "eval" / "ifbench.parquet",
    data_root / "if" / "eval" / "ifeval.parquet",
    *lcb_paths,
]
incomplete = [str(path) for path in expected_eval if not parquet_complete(path)]
if incomplete:
    raise RuntimeError("Incomplete evaluation files: " + ", ".join(incomplete))

summary = {
    "status": "complete",
    "data_root": str(data_root),
    "training_rows_per_domain": int(TRAIN_ROWS),
    "note": "FedAvg_mopsd.ipynb used the first 2000 rows of each deterministic 3000-row file.",
    "training": {domain: str(path) for domain, path in expected_training.items()},
    "evaluation_files": [str(path) for path in expected_eval],
    "completed_at_utc": utc_now(),
}
write_manifest(data_root / "manifest.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
